# SemCons-IDS — Pipeline final exécutable

**Context-Aware Semantic Consistency Fine-Tuning of Lightweight Language Models for Reliable and Explainable Intrusion Detection**

Ce notebook implémente le pipeline complet :

1. chargement universel de CICIDS2017 (`.parquet` ou `.csv`) ;
2. normalisation robuste des labels et déduplication ;
3. séparation source-aware en `train / validation / calibration / test` avant tout ajustement ;
4. imputation, sélection RF Top-k et standardisation ajustées sur le train uniquement ;
5. sérialisation sémantique contextuelle à deux vues ;
6. MiniLM + LoRA par défaut, avec ModernBERT et DeBERTa activables individuellement ;
7. perte `classification + λ × Jensen–Shannon` ;
8. temperature scaling, prédiction conforme APS et abstention ;
9. robustesse aux sérialisations, explicabilité par occlusion et métriques de frugalité.

> Exécuter les cellules dans l’ordre. Pour une vérification rapide, remplacer `RUN_MODE = "article"` par `RUN_MODE = "quick"` dans la configuration.

## 0 — Installation

La cellule n’installe pas PyTorch afin de préserver la version CUDA fournie par Colab. Après une mise à jour importante de bibliothèques dans une session déjà active, redémarrer l’environnement avant de continuer.

In [ ]:
import os
import sys
import signal
import subprocess
import importlib.metadata as metadata
import importlib.util

PINNED = {
    "transformers": "4.57.6",
    "peft": "0.18.0",
    "accelerate": "1.12.0",
}

def package_version(name: str):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None

def hf_stack_is_importable() -> tuple[bool, str]:
    """Teste la pile HF dans un nouveau processus, sans polluer ce noyau."""
    code = (
        "from transformers import AutoTokenizer, AutoModelForSequenceClassification;"
        "from peft import LoraConfig, TaskType, get_peft_model;"
        "print('HF_STACK_OK')"
    )
    result = subprocess.run(
        [sys.executable, "-c", code],
        text=True,
        capture_output=True,
    )
    details = (result.stdout + "\n" + result.stderr).strip()
    return result.returncode == 0 and "HF_STACK_OK" in result.stdout, details

installed = {name: package_version(name) for name in PINNED}
torchao_version = package_version("torchao")
stack_ok, stack_details = hf_stack_is_importable()

needs_repair = (
    torchao_version is not None
    or any(installed[name] != version for name, version in PINNED.items())
    or not stack_ok
)

print("Versions avant vérification :")
for name, expected in PINNED.items():
    print(f"  {name:14s}: {installed[name]}  (attendu : {expected})")
print(f"  {'torchao':14s}: {torchao_version or 'absent'}")
print(f"  {'imports HF':14s}: {'OK' if stack_ok else 'ÉCHEC'}")

if needs_repair:
    print("\nRéparation propre de l'environnement Hugging Face...")

    # Désinstallation préalable : évite les fichiers résiduels de versions 4.x/5.x.
    subprocess.run(
        [
            sys.executable, "-m", "pip", "uninstall", "-y",
            "transformers", "peft", "accelerate", "tokenizers", "torchao",
        ],
        check=False,
    )

    subprocess.check_call(
        [
            sys.executable, "-m", "pip", "install",
            "--no-cache-dir",
            "transformers==4.57.6",
            "peft==0.18.0",
            "accelerate==1.12.0",
        ]
    )

    # Installer uniquement les bibliothèques auxiliaires réellement absentes.
    optional_modules = {
        "imblearn": "imbalanced-learn>=0.12.0",
        "pyarrow": "pyarrow>=15.0.0",
        "joblib": "joblib>=1.3.0",
        "psutil": "psutil>=5.9.0",
    }
    missing = [
        requirement
        for module, requirement in optional_modules.items()
        if importlib.util.find_spec(module) is None
    ]
    if missing:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "--no-cache-dir", *missing]
        )

    repaired_versions = {name: package_version(name) for name in PINNED}
    repaired_ok, repaired_details = hf_stack_is_importable()

    print("\nVersions après réparation :")
    for name, expected in PINNED.items():
        print(f"  {name:14s}: {repaired_versions[name]}  (attendu : {expected})")

    if not repaired_ok:
        raise RuntimeError(
            "La réinstallation s'est terminée, mais le test d'importation a échoué.\n"
            + repaired_details
        )

    print("\n✅ Installation réparée.")
    print("♻️ Redémarrage automatique du noyau pour charger les nouveaux paquets...")
    os.kill(os.getpid(), signal.SIGKILL)

# Aucun paquet HF n'est importé dans cette cellule lorsque l'environnement
# vient d'être modifié. Au second passage, le test s'effectue dans un
# sous-processus propre et l'exécution continue normalement.
print("\n✅ Environnement stable et prêt.")

## 1 — Imports, reproductibilité et configuration

Un seul backbone est actif par exécution : `minilm`, `modernbert` ou `deberta`.

In [ ]:
from __future__ import annotations

import os
import re
import gc
import json
import math
import time
import random
import warnings
import importlib.metadata as metadata
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Set, Tuple

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from matplotlib import pyplot as plt
from peft import LoraConfig, TaskType, get_peft_model
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
    matthews_corrcoef,
    precision_score,
    recall_score,
)
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import DataLoader, Dataset, Subset
from tqdm.auto import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer

warnings.filterwarnings("ignore")

EXPECTED_HF_VERSIONS = {
    "transformers": "4.57.6",
    "peft": "0.18.0",
    "accelerate": "1.12.0",
}
for _package, _expected in EXPECTED_HF_VERSIONS.items():
    _found = metadata.version(_package)
    if _found != _expected:
        raise RuntimeError(
            f"Version incompatible pour {_package}: {_found}; attendu: {_expected}. "
            "Réexécutez la cellule d'installation, laissez le noyau redémarrer, "
            "puis lancez toutes les cellules depuis le début."
        )
print("Pile Hugging Face :", " | ".join(
    f"{name}={metadata.version(name)}" for name in EXPECTED_HF_VERSIONS
))

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    print("Exécution hors Colab : Google Drive non monté.")

@dataclass
class ExperimentConfig:
    data_dir: str = "/content/drive/MyDrive/Dataset/CIC-IDS-2017/"
    output_dir: str = "/content/drive/MyDrive/these_llm/SemCons_IDS_CICIDS2017/"
    run_mode: str = "article"  # "quick" ou "article"
    backbone: str = "minilm"   # minilm | modernbert | deberta
    seed: int = 42

    top_k_features: int = 20
    max_length: int = 256
    rf_selection_rows: int = 200_000
    min_samples_per_class: int = 200

    train_fraction: float = 0.70
    val_fraction: float = 0.10
    cal_fraction: float = 0.10
    test_fraction: float = 0.10

    use_class_weights: bool = True
    use_random_oversampling: bool = False
    train_cap_per_class: Optional[int] = 10_000
    eval_cap_per_class: Optional[int] = None

    lora_r: int = 32
    lora_alpha: int = 64
    lora_dropout: float = 0.10
    lambda_consistency: float = 0.50

    epochs: int = 5
    learning_rate: float = 2e-4
    weight_decay: float = 0.01
    batch_train: int = 16
    batch_eval: int = 64
    grad_accum: int = 2
    warmup_fraction: float = 0.10
    max_grad_norm: float = 1.0
    early_stopping_patience: int = 2

    conformal_alpha: float = 0.10
    robustness_samples: int = 5_000
    explanation_samples: int = 100
    explanation_top_k: int = 5
    run_numeric_baseline: bool = True

CFG = ExperimentConfig()

# Profil rapide : vérification de bout en bout sur un volume réduit.
if CFG.run_mode == "quick":
    CFG.train_cap_per_class = 2_000
    CFG.eval_cap_per_class = 1_000
    CFG.epochs = 2
    CFG.robustness_samples = 1_000
    CFG.explanation_samples = 20

BACKBONES = {
    "minilm": {
        "checkpoint": "sentence-transformers/all-MiniLM-L6-v2",
        "targets": ["query", "value"],
        "modules_to_save": ["classifier"],
    },
    "modernbert": {
        "checkpoint": "answerdotai/ModernBERT-base",
        "targets": ["Wqkv"],
        "modules_to_save": ["classifier"],
    },
    "deberta": {
        "checkpoint": "microsoft/deberta-v3-small",
        "targets": ["query_proj", "value_proj"],
        "modules_to_save": ["classifier", "pooler"],
    },
}

if CFG.backbone not in BACKBONES:
    raise ValueError(f"Backbone inconnu : {CFG.backbone}")
MODEL_SPEC = BACKBONES[CFG.backbone]
MODEL_CHECKPOINT = MODEL_SPEC["checkpoint"]

TARGET_CLASSES = [
    "BENIGN", "Bot", "DDoS", "DoS_GoldenEye", "DoS_Hulk",
    "DoS_Slowhttptest", "DoS_Slowloris", "FTP-Patator",
    "PortScan", "SSH-Patator",
]

Path(CFG.output_dir).mkdir(parents=True, exist_ok=True)


def set_seed(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if torch.backends.cudnn.is_available():
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


set_seed(CFG.seed)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(json.dumps(asdict(CFG), indent=2, ensure_ascii=False))
print(f"\nBackbone : {MODEL_CHECKPOINT}")
print(f"Device   : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM     : {torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GiB")

## 2 — Chargement universel et normalisation des labels

La colonne `_source_file` est conservée afin de créer une séparation source-aware. Les labels sont harmonisés sans tenir compte de la casse, des espaces ou des tirets Unicode.

In [ ]:
LABEL_MAP = {
    "benign": "BENIGN",
    "bot": "Bot",
    "ddos": "DDoS",
    "dos hulk": "DoS_Hulk",
    "dos goldeneye": "DoS_GoldenEye",
    "dos slowhttptest": "DoS_Slowhttptest",
    "dos slowloris": "DoS_Slowloris",
    "ftp patator": "FTP-Patator",
    "ssh patator": "SSH-Patator",
    "portscan": "PortScan",
    "port scan": "PortScan",
    "infiltration": "Infiltration",
    "heartbleed": "Heartbleed",
    "web attack brute force": "Web_Attack_Brute_Force",
    "web attack xss": "Web_Attack_XSS",
    "web attack sql injection": "Web_Attack_Sql_Injection",
}


def normalize_column_name(name: str) -> str:
    name = str(name).strip().lower()
    name = re.sub(r"[\s/\\-]+", "_", name)
    name = re.sub(r"[^a-z0-9_]", "", name)
    return re.sub(r"_+", "_", name).strip("_")


def normalize_label_key(value: object) -> str:
    text = str(value).strip().replace("�", "-").replace("–", "-").replace("—", "-")
    text = text.replace("_", " ").replace("-", " ")
    return re.sub(r"\s+", " ", text).strip().lower()


def read_table(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()
    if suffix == ".parquet":
        return pd.read_parquet(path)
    if suffix == ".csv":
        return pd.read_csv(path, low_memory=False)
    if suffix in {".feather", ".ftr"}:
        return pd.read_feather(path)
    raise ValueError(f"Format non pris en charge : {path}")


data_dir = Path(CFG.data_dir)
files = sorted([
    *data_dir.glob("*.parquet"),
    *data_dir.glob("*.csv"),
    *data_dir.glob("*.feather"),
    *data_dir.glob("*.ftr"),
])
if not files:
    raise FileNotFoundError(f"Aucun fichier CSV/Parquet/Feather trouvé dans {data_dir}")

frames: List[pd.DataFrame] = []
for path in files:
    part = read_table(path)
    part.columns = [normalize_column_name(c) for c in part.columns]
    part["_source_file"] = path.name
    part["_source_row"] = np.arange(len(part), dtype=np.int64)
    frames.append(part)
    print(f"{path.name:<50s} {len(part):>10,} lignes")

df = pd.concat(frames, ignore_index=True, sort=False)
del frames
gc.collect()

label_candidates = [c for c in df.columns if c in {"label", "class", "attack", "attack_cat"}]
if not label_candidates:
    label_candidates = [c for c in df.columns if "label" in c or "class" in c]
if not label_candidates:
    raise ValueError("Aucune colonne de label détectée.")
LABEL_COL = label_candidates[0]

raw_labels = df[LABEL_COL].copy()
df[LABEL_COL] = raw_labels.map(lambda x: LABEL_MAP.get(normalize_label_key(x), np.nan))
unmapped = df[LABEL_COL].isna()
if unmapped.any():
    examples = raw_labels[unmapped].astype(str).value_counts().head(15)
    print("\nLabels non reconnus supprimés :")
    print(examples)
    df = df.loc[~unmapped].copy()

# Tâche principale définie dans la configuration scientifique.
df = df[df[LABEL_COL].isin(TARGET_CLASSES)].copy()
if "BENIGN" not in set(df[LABEL_COL]):
    raise ValueError("BENIGN est absente. Vérifier le fichier Monday/Benign et le mappage des labels.")

# Conversion explicite des colonnes candidates : les textes non convertibles deviennent NaN.
metadata_cols = {LABEL_COL, "_source_file", "_source_row"}
for col in [c for c in df.columns if c not in metadata_cols]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

feature_candidates = [c for c in df.columns if c not in metadata_cols and pd.api.types.is_numeric_dtype(df[c])]
if not feature_candidates:
    raise ValueError("Aucune caractéristique numérique détectée.")

df[feature_candidates] = df[feature_candidates].replace([np.inf, -np.inf], np.nan)

# Déduplication sur le contenu du flux et le label, sans utiliser les métadonnées de provenance.
n_before = len(df)
df = df.drop_duplicates(subset=feature_candidates + [LABEL_COL], keep="first").reset_index(drop=True)
print(f"\nDoublons supprimés : {n_before - len(df):,}")
print(f"Dataset retenu     : {len(df):,} lignes × {len(feature_candidates)} variables numériques")
print(df[LABEL_COL].value_counts())

## 3 — Split source-aware avant le prétraitement

Chaque groupe `(fichier source, classe)` est ordonné selon sa position dans le fichier puis réparti en 70/10/10/10. Cela empêche le prétraitement et la sélection des variables de voir les partitions futures, tout en gardant chaque classe représentée lorsque son effectif le permet.

In [ ]:
def split_group_indices(n: int, fractions: Sequence[float]) -> Tuple[slice, slice, slice, slice]:
    if n < 4:
        raise ValueError("Un groupe conservé doit contenir au moins quatre lignes.")
    raw = np.asarray(fractions, dtype=float) * n
    counts = np.floor(raw).astype(int)
    counts[counts == 0] = 1
    while counts.sum() > n:
        j = int(np.argmax(counts))
        if counts[j] > 1:
            counts[j] -= 1
        else:
            break
    while counts.sum() < n:
        residuals = raw - np.floor(raw)
        counts[int(np.argmax(residuals))] += 1
    a, b, c, d = counts.tolist()
    return slice(0, a), slice(a, a+b), slice(a+b, a+b+c), slice(a+b+c, a+b+c+d)


def source_aware_split(frame: pd.DataFrame) -> Dict[str, pd.DataFrame]:
    fractions = [CFG.train_fraction, CFG.val_fraction, CFG.cal_fraction, CFG.test_fraction]
    if not np.isclose(sum(fractions), 1.0):
        raise ValueError("Les fractions de split doivent totaliser 1.")

    buckets: Dict[str, List[pd.DataFrame]] = {"train": [], "val": [], "cal": [], "test": []}
    for (_, label), group in frame.groupby(["_source_file", LABEL_COL], sort=False):
        group = group.sort_values("_source_row")
        if len(group) < 4:
            print(f"Groupe ignoré (<4) : {label} — {len(group)}")
            continue
        slices = split_group_indices(len(group), fractions)
        for key, slc in zip(buckets, slices):
            buckets[key].append(group.iloc[slc])

    return {
        key: pd.concat(parts, ignore_index=True).sample(frac=1.0, random_state=CFG.seed).reset_index(drop=True)
        for key, parts in buckets.items()
    }


def cap_per_class(frame: pd.DataFrame, cap: Optional[int], seed: int) -> pd.DataFrame:
    if cap is None:
        return frame.reset_index(drop=True)
    parts = []
    for label, group in frame.groupby(LABEL_COL, sort=False):
        parts.append(group.sample(n=min(len(group), cap), random_state=seed))
    return pd.concat(parts, ignore_index=True).sample(frac=1.0, random_state=seed).reset_index(drop=True)

splits = source_aware_split(df)

# Le cap d'entraînement est une réduction de coût. Les partitions d'évaluation restent naturelles en mode article.
splits["train"] = cap_per_class(splits["train"], CFG.train_cap_per_class, CFG.seed)
for key in ["val", "cal", "test"]:
    splits[key] = cap_per_class(splits[key], CFG.eval_cap_per_class, CFG.seed + 1)

for key, frame in splits.items():
    print(f"\n{key.upper():5s} : {len(frame):,} lignes")
    print(frame[LABEL_COL].value_counts().sort_index())

expected = set(TARGET_CLASSES)
for key, frame in splits.items():
    missing = expected - set(frame[LABEL_COL])
    if missing:
        raise ValueError(f"Classes absentes de {key}: {sorted(missing)}")

## 4 — Prétraitement ajusté uniquement sur le train

Ordre appliqué : imputation médiane → suppression des variables constantes → sélection RF Top-k → standardisation. Les objets ajustés sont sauvegardés pour la reproductibilité.

In [ ]:
X_train_df = splits["train"][feature_candidates]

imputer = SimpleImputer(strategy="median")
X_train_imp_all = imputer.fit_transform(X_train_df).astype(np.float32)

variance_filter = VarianceThreshold(threshold=0.0)
X_train_var = variance_filter.fit_transform(X_train_imp_all).astype(np.float32)
var_features = [f for f, keep in zip(feature_candidates, variance_filter.get_support()) if keep]

label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(splits["train"][LABEL_COL])
class_names = label_encoder.classes_.tolist()
N_CLASSES = len(class_names)

rf_rows = min(CFG.rf_selection_rows, len(X_train_var))
rng = np.random.default_rng(CFG.seed)
rf_idx = rng.choice(len(X_train_var), size=rf_rows, replace=False)
rf_selector = RandomForestClassifier(
    n_estimators=200,
    max_depth=14,
    class_weight="balanced_subsample",
    n_jobs=-1,
    random_state=CFG.seed,
)
rf_selector.fit(X_train_var[rf_idx], y_train[rf_idx])

importance_df = pd.DataFrame({
    "feature": var_features,
    "importance": rf_selector.feature_importances_,
}).sort_values("importance", ascending=False).reset_index(drop=True)

SELECTED_FEATURES = importance_df.head(CFG.top_k_features)["feature"].tolist()
selected_positions = [var_features.index(f) for f in SELECTED_FEATURES]

scaler = StandardScaler()
X_train_raw = X_train_var[:, selected_positions].astype(np.float32)
X_train_scaled = scaler.fit_transform(X_train_raw).astype(np.float32)


def transform_split(frame: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    X_imp = imputer.transform(frame[feature_candidates]).astype(np.float32)
    X_var = variance_filter.transform(X_imp).astype(np.float32)
    X_raw = X_var[:, selected_positions].astype(np.float32)
    X_scaled = scaler.transform(X_raw).astype(np.float32)
    y = label_encoder.transform(frame[LABEL_COL]).astype(np.int64)
    return X_raw, X_scaled, y

arrays: Dict[str, Dict[str, np.ndarray]] = {
    "train": {"raw": X_train_raw, "scaled": X_train_scaled, "y": y_train.astype(np.int64)}
}
for key in ["val", "cal", "test"]:
    raw, scaled, labels = transform_split(splits[key])
    arrays[key] = {"raw": raw, "scaled": scaled, "y": labels}

# Quantiles sémantiques calculés avant tout éventuel suréchantillonnage.
SEMANTIC_QUANTILES = {
    feature: np.quantile(X_train_raw[:, j], [0.10, 0.25, 0.75, 0.90]).astype(float).tolist()
    for j, feature in enumerate(SELECTED_FEATURES)
}

if CFG.use_random_oversampling:
    from imblearn.over_sampling import RandomOverSampler
    ros = RandomOverSampler(random_state=CFG.seed)
    indices = np.arange(len(y_train)).reshape(-1, 1)
    indices_resampled, y_resampled = ros.fit_resample(indices, y_train)
    idx = indices_resampled.ravel()
    arrays["train"] = {
        "raw": arrays["train"]["raw"][idx],
        "scaled": arrays["train"]["scaled"][idx],
        "y": y_resampled.astype(np.int64),
    }

if CFG.use_class_weights:
    classes = np.arange(N_CLASSES)
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=arrays["train"]["y"])
    CLASS_WEIGHTS = torch.tensor(weights, dtype=torch.float32)
else:
    CLASS_WEIGHTS = torch.ones(N_CLASSES, dtype=torch.float32)

print("\nTop caractéristiques :")
display(importance_df.head(CFG.top_k_features))
print("\nClasses :", class_names)
print("Poids   :", dict(zip(class_names, CLASS_WEIGHTS.numpy().round(3))))

artifact_dir = Path(CFG.output_dir) / "preprocessing"
artifact_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(imputer, artifact_dir / "imputer.joblib")
joblib.dump(variance_filter, artifact_dir / "variance_filter.joblib")
joblib.dump(rf_selector, artifact_dir / "rf_selector.joblib")
joblib.dump(scaler, artifact_dir / "scaler.joblib")
joblib.dump(label_encoder, artifact_dir / "label_encoder.joblib")
joblib.dump(SELECTED_FEATURES, artifact_dir / "selected_features.joblib")
with open(artifact_dir / "semantic_quantiles.json", "w", encoding="utf-8") as f:
    json.dump(SEMANTIC_QUANTILES, f, indent=2)
importance_df.to_csv(artifact_dir / "feature_importance.csv", index=False)

## 5 — Sérialisation sémantique contextuelle

La vue canonique garde un ordre fixe. La vue perturbée change uniquement la forme : ordre des groupes, ordre des variables, synonymes contrôlés et précision numérique. Le sens du flux et le label restent inchangés.

In [ ]:
TOKEN_REPLACEMENTS = {
    "fwd": "forward", "bwd": "backward", "pkt": "packet", "pkts": "packets",
    "len": "length", "avg": "average", "std": "standard deviation",
    "iat": "inter-arrival time", "cnt": "count", "tot": "total",
    "win": "window", "seg": "segment", "hdr": "header",
    "init": "initial", "min": "minimum", "max": "maximum",
}

GROUP_ORDER = ["Rate", "Timing", "Directionality", "Packet size", "TCP state", "Volume", "Other"]


def humanize_feature(name: str) -> str:
    tokens = [TOKEN_REPLACEMENTS.get(tok, tok) for tok in name.lower().split("_")]
    return " ".join(tokens).replace("  ", " ").strip()


def feature_group(name: str) -> str:
    n = name.lower()
    if any(k in n for k in ["packets_s", "bytes_s", "packet_rate", "flow_rate"]):
        return "Rate"
    if any(k in n for k in ["iat", "duration", "active", "idle"]):
        return "Timing"
    if any(k in n for k in ["fwd", "bwd", "forward", "backward", "subflow"]):
        return "Directionality"
    if any(k in n for k in ["length", "size", "segment"]):
        return "Packet size"
    if any(k in n for k in ["flag", "window", "header", "syn", "ack", "rst", "psh", "urg"]):
        return "TCP state"
    if any(k in n for k in ["byte", "packet", "count", "total"]):
        return "Volume"
    return "Other"


def semantic_state(value: float, quantiles: Sequence[float]) -> str:
    q10, q25, q75, q90 = quantiles
    if value <= q10:
        return "very low"
    if value <= q25:
        return "low"
    if value < q75:
        return "typical"
    if value < q90:
        return "high"
    return "very high"


STATE_VARIANTS = {
    "very low": ["very low", "far below typical", "unusually low"],
    "low": ["low", "below typical", "moderately low"],
    "typical": ["typical", "within the usual range", "ordinary"],
    "high": ["high", "above typical", "elevated"],
    "very high": ["very high", "far above typical", "unusually high"],
}


def find_feature_index(required_terms: Sequence[str]) -> Optional[int]:
    for j, feature in enumerate(SELECTED_FEATURES):
        low = feature.lower()
        if all(term in low for term in required_terms):
            return j
    return None


FWD_TOTAL_IDX = find_feature_index(["fwd", "total"])
BWD_TOTAL_IDX = find_feature_index(["bwd", "total"])
FLOW_RATE_IDX = next((j for j, f in enumerate(SELECTED_FEATURES) if "packets_s" in f.lower()), None)
DURATION_IDX = next((j for j, f in enumerate(SELECTED_FEATURES) if "duration" in f.lower()), None)


class SemanticSerializer:
    def __init__(self, features: Sequence[str], quantiles: Dict[str, Sequence[float]], seed: int = 42):
        self.features = list(features)
        self.quantiles = quantiles
        self.seed = seed
        self.group_to_indices: Dict[str, List[int]] = {g: [] for g in GROUP_ORDER}
        for j, feature in enumerate(self.features):
            self.group_to_indices[feature_group(feature)].append(j)

    def _relations(self, raw: np.ndarray) -> List[str]:
        relations: List[str] = []
        if FWD_TOTAL_IDX is not None and BWD_TOTAL_IDX is not None:
            fwd = max(float(raw[FWD_TOTAL_IDX]), 0.0)
            bwd = max(float(raw[BWD_TOTAL_IDX]), 0.0)
            ratio = (fwd + 1e-6) / (bwd + 1e-6)
            if ratio >= 2.0:
                relations.append("Forward traffic strongly dominates backward traffic.")
            elif ratio <= 0.5:
                relations.append("Backward traffic strongly dominates forward traffic.")
            else:
                relations.append("Forward and backward traffic volumes are broadly balanced.")
        if FLOW_RATE_IDX is not None and DURATION_IDX is not None:
            rate_state = semantic_state(float(raw[FLOW_RATE_IDX]), self.quantiles[self.features[FLOW_RATE_IDX]])
            duration_state = semantic_state(float(raw[DURATION_IDX]), self.quantiles[self.features[DURATION_IDX]])
            if rate_state in {"high", "very high"} and duration_state in {"low", "very low"}:
                relations.append("High packet activity occurs during a short-lived flow.")
        return relations

    def build(
        self,
        raw: np.ndarray,
        scaled: np.ndarray,
        sample_id: int,
        perturbed: bool = False,
        omit_features: Optional[Set[int]] = None,
    ) -> str:
        omit_features = omit_features or set()
        rng = np.random.default_rng(self.seed + int(sample_id) * 104729 + (1 if perturbed else 0))

        groups = [g for g in GROUP_ORDER if self.group_to_indices[g]]
        if perturbed:
            rng.shuffle(groups)

        sections: List[str] = ["Network flow summary."]
        for group in groups:
            indices = [j for j in self.group_to_indices[group] if j not in omit_features]
            if not indices:
                continue
            if perturbed:
                rng.shuffle(indices)
            sentences: List[str] = []
            for j in indices:
                feature = self.features[j]
                state = semantic_state(float(raw[j]), self.quantiles[feature])
                state_text = rng.choice(STATE_VARIANTS[state]) if perturbed else state
                precision = 2 if perturbed else 3
                value_text = f"{float(scaled[j]):.{precision}f}"
                description = humanize_feature(feature)
                if perturbed and rng.random() < 0.5:
                    sentence = f"{description.capitalize()} is {state_text} [z={value_text}]."
                else:
                    sentence = f"The {description} is {state_text} [z={value_text}]."
                sentences.append(sentence)
            sections.append(f"{group}: " + " ".join(sentences))

        relations = self._relations(raw)
        if relations:
            if perturbed:
                rng.shuffle(relations)
            sections.append("Relations: " + " ".join(relations))
        return " ".join(sections)


serializer = SemanticSerializer(SELECTED_FEATURES, SEMANTIC_QUANTILES, seed=CFG.seed)
example_id = 0
print("VUE CANONIQUE\n", serializer.build(arrays["train"]["raw"][example_id], arrays["train"]["scaled"][example_id], example_id))
print("\nVUE PERTURBÉE\n", serializer.build(arrays["train"]["raw"][example_id], arrays["train"]["scaled"][example_id], example_id, perturbed=True))

## 6 — Dataset indexé et collateur à double vue

Les textes sont construits et tokenisés par lot dans le collateur. Cette stratégie évite de stocker deux matrices tokenisées complètes en mémoire.

In [ ]:
HF_TOKEN = os.getenv("HF_TOKEN")
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT, token=HF_TOKEN)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token or tokenizer.sep_token


class IndexDataset(Dataset):
    def __init__(self, labels: np.ndarray):
        self.labels = np.asarray(labels, dtype=np.int64)

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, idx: int) -> Dict[str, int]:
        return {"index": int(idx), "label": int(self.labels[idx])}


class SemanticBatchCollator:
    def __init__(
        self,
        tokenizer,
        raw: np.ndarray,
        scaled: np.ndarray,
        serializer: SemanticSerializer,
        max_length: int,
        dual_view: bool,
    ):
        self.tokenizer = tokenizer
        self.raw = raw
        self.scaled = scaled
        self.serializer = serializer
        self.max_length = max_length
        self.dual_view = dual_view

    def __call__(self, items: List[Dict[str, int]]) -> Dict[str, torch.Tensor]:
        indices = [item["index"] for item in items]
        labels = torch.tensor([item["label"] for item in items], dtype=torch.long)
        texts_1 = [
            self.serializer.build(self.raw[i], self.scaled[i], sample_id=i, perturbed=False)
            for i in indices
        ]
        enc_1 = self.tokenizer(
            texts_1,
            padding=True,
            truncation=True,
            max_length=self.max_length,
            pad_to_multiple_of=8,
            return_tensors="pt",
        )
        batch = {
            "input_ids_1": enc_1["input_ids"],
            "attention_mask_1": enc_1["attention_mask"],
            "labels": labels,
            "indices": torch.tensor(indices, dtype=torch.long),
        }
        if self.dual_view:
            texts_2 = [
                self.serializer.build(self.raw[i], self.scaled[i], sample_id=i, perturbed=True)
                for i in indices
            ]
            enc_2 = self.tokenizer(
                texts_2,
                padding=True,
                truncation=True,
                max_length=self.max_length,
                pad_to_multiple_of=8,
                return_tensors="pt",
            )
            batch["input_ids_2"] = enc_2["input_ids"]
            batch["attention_mask_2"] = enc_2["attention_mask"]
        return batch


def make_loader(split: str, batch_size: int, shuffle: bool, dual_view: bool) -> DataLoader:
    dataset = IndexDataset(arrays[split]["y"])
    collator = SemanticBatchCollator(
        tokenizer=tokenizer,
        raw=arrays[split]["raw"],
        scaled=arrays[split]["scaled"],
        serializer=serializer,
        max_length=CFG.max_length,
        dual_view=dual_view,
    )
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0,
        pin_memory=(DEVICE.type == "cuda"),
        collate_fn=collator,
    )


train_loader = make_loader("train", CFG.batch_train, shuffle=True, dual_view=True)
val_loader = make_loader("val", CFG.batch_eval, shuffle=False, dual_view=False)
cal_loader = make_loader("cal", CFG.batch_eval, shuffle=False, dual_view=False)
test_loader = make_loader("test", CFG.batch_eval, shuffle=False, dual_view=False)

preview = next(iter(train_loader))
print({k: tuple(v.shape) for k, v in preview.items()})

## 7 — Backbone + LoRA

Le notebook affiche séparément les paramètres LoRA et les autres modules entraînables. Pour DeBERTa, le pooler nouvellement initialisé est également sauvegardé et entraîné.

In [ ]:
# Contrôle préventif : torchao n'est pas utilisé par ce pipeline.
try:
    import importlib.metadata as _metadata
    _torchao_version = _metadata.version("torchao")
except _metadata.PackageNotFoundError:
    _torchao_version = None

if _torchao_version is not None:
    raise RuntimeError(
        f"torchao {_torchao_version} est encore présent dans l'environnement. "
        "Exécutez la cellule d'installation au début du notebook, "
        "puis redémarrez l'environnement Colab si cette session avait déjà importé PEFT."
    )

base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=N_CLASSES,
    id2label={i: name for i, name in enumerate(class_names)},
    label2id={name: i for i, name in enumerate(class_names)},
    ignore_mismatched_sizes=True,
    token=HF_TOKEN,
)

linear_leaf_names = sorted({
    name.split(".")[-1]
    for name, module in base_model.named_modules()
    if isinstance(module, nn.Linear)
})
print("Modules linéaires disponibles :", linear_leaf_names)

missing_targets = [target for target in MODEL_SPEC["targets"] if target not in linear_leaf_names]
if missing_targets:
    raise ValueError(
        f"Cibles LoRA absentes pour {CFG.backbone}: {missing_targets}. "
        f"Modules disponibles: {linear_leaf_names}"
    )

# Ne garder que les modules réellement présents ; évite un échec si une architecture change légèrement.
all_module_names = [name for name, _ in base_model.named_modules()]
modules_to_save = [
    module_name for module_name in MODEL_SPEC["modules_to_save"]
    if any(name == module_name or name.endswith("." + module_name) for name in all_module_names)
]

lora_config = LoraConfig(
    r=CFG.lora_r,
    lora_alpha=CFG.lora_alpha,
    lora_dropout=CFG.lora_dropout,
    target_modules=MODEL_SPEC["targets"],
    modules_to_save=modules_to_save,
    bias="none",
    task_type=TaskType.SEQ_CLS,
)
model = get_peft_model(base_model, lora_config).to(DEVICE)

n_total = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_lora = sum(p.numel() for name, p in model.named_parameters() if p.requires_grad and "lora_" in name)
n_head = n_trainable - n_lora

print(f"Total                 : {n_total:,}")
print(f"LoRA entraînable      : {n_lora:,} ({100*n_lora/n_total:.4f}%)")
print(f"Tête/pooler entraînable: {n_head:,} ({100*n_head/n_total:.4f}%)")
print(f"Total entraînable     : {n_trainable:,} ({100*n_trainable/n_total:.4f}%)")
print(f"Modules sauvegardés   : {modules_to_save}")

## 8 — Fonctions de perte et d’évaluation

In [ ]:
def js_divergence(logits_1: torch.Tensor, logits_2: torch.Tensor) -> torch.Tensor:
    p = F.softmax(logits_1, dim=-1)
    q = F.softmax(logits_2, dim=-1)
    m = 0.5 * (p + q)
    eps = 1e-8
    kl_pm = torch.sum(p * (torch.log(p + eps) - torch.log(m + eps)), dim=-1)
    kl_qm = torch.sum(q * (torch.log(q + eps) - torch.log(m + eps)), dim=-1)
    return 0.5 * (kl_pm + kl_qm).mean()


def ece_score(probs: np.ndarray, labels: np.ndarray, n_bins: int = 15) -> float:
    confidences = probs.max(axis=1)
    predictions = probs.argmax(axis=1)
    correct = (predictions == labels).astype(float)
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for left, right in zip(edges[:-1], edges[1:]):
        mask = (confidences > left) & (confidences <= right)
        if mask.any():
            ece += mask.mean() * abs(correct[mask].mean() - confidences[mask].mean())
    return float(ece)


def brier_multiclass(probs: np.ndarray, labels: np.ndarray, n_classes: int) -> float:
    one_hot = np.eye(n_classes, dtype=np.float32)[labels]
    return float(np.mean(np.sum((probs - one_hot) ** 2, axis=1)))


def compute_metrics_from_logits(logits: np.ndarray, labels: np.ndarray) -> Dict[str, float]:
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    preds = probs.argmax(axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
        "weighted_f1": f1_score(labels, preds, average="weighted", zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(labels, preds),
        "mcc": matthews_corrcoef(labels, preds),
        "macro_precision": precision_score(labels, preds, average="macro", zero_division=0),
        "macro_recall": recall_score(labels, preds, average="macro", zero_division=0),
        "nll": log_loss(labels, probs, labels=np.arange(N_CLASSES)),
        "brier": brier_multiclass(probs, labels, N_CLASSES),
        "ece": ece_score(probs, labels),
    }


@torch.no_grad()
def collect_logits(loader: DataLoader) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    model.eval()
    logits_all, labels_all, indices_all = [], [], []
    for batch in tqdm(loader, desc="Inférence", leave=False):
        input_ids = batch["input_ids_1"].to(DEVICE, non_blocking=True)
        attention_mask = batch["attention_mask_1"].to(DEVICE, non_blocking=True)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits_all.append(outputs.logits.float().cpu().numpy())
        labels_all.append(batch["labels"].numpy())
        indices_all.append(batch["indices"].numpy())
    return np.concatenate(logits_all), np.concatenate(labels_all), np.concatenate(indices_all)

## 9 — Entraînement SemCons

Les deux vues sont concaténées dans un seul passage avant afin de réduire le coût. La perte finale est :

\[
\mathcal{L}=	frac{1}{2}[CE(p_1,y)+CE(p_2,y)] + \lambda D_{JS}(p_1,p_2).
\]

In [ ]:
criterion = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS.to(DEVICE))
optimizer = AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=CFG.learning_rate,
    weight_decay=CFG.weight_decay,
)
steps_per_epoch = math.ceil(len(train_loader) / CFG.grad_accum)
total_steps = max(1, CFG.epochs * steps_per_epoch)
warmup_steps = int(CFG.warmup_fraction * total_steps)
def linear_warmup_decay_lambda(current_step: int) -> float:
    """Warmup linéaire, puis décroissance linéaire jusqu'à zéro."""
    if current_step < warmup_steps:
        return float(current_step) / float(max(1, warmup_steps))
    remaining_steps = total_steps - current_step
    decay_steps = total_steps - warmup_steps
    return max(0.0, float(remaining_steps) / float(max(1, decay_steps)))

scheduler = LambdaLR(optimizer, lr_lambda=linear_warmup_decay_lambda)
print(
    f"Scheduler PyTorch : {warmup_steps:,} étapes de warmup / "
    f"{total_steps:,} étapes d'optimisation"
)

amp_enabled = DEVICE.type == "cuda"
grad_scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)

best_macro_f1 = -np.inf
best_state = None
patience_counter = 0
history: List[Dict[str, float]] = []

if DEVICE.type == "cuda":
    torch.cuda.reset_peak_memory_stats()
train_start = time.perf_counter()

for epoch in range(1, CFG.epochs + 1):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    running_loss = 0.0
    running_cls = 0.0
    running_js = 0.0

    progress = tqdm(train_loader, desc=f"Epoch {epoch}/{CFG.epochs}")
    for step, batch in enumerate(progress, start=1):
        ids_1 = batch["input_ids_1"].to(DEVICE, non_blocking=True)
        mask_1 = batch["attention_mask_1"].to(DEVICE, non_blocking=True)
        ids_2 = batch["input_ids_2"].to(DEVICE, non_blocking=True)
        mask_2 = batch["attention_mask_2"].to(DEVICE, non_blocking=True)
        labels = batch["labels"].to(DEVICE, non_blocking=True)

        ids = torch.cat([ids_1, ids_2], dim=0)
        masks = torch.cat([mask_1, mask_2], dim=0)

        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=amp_enabled):
            logits = model(input_ids=ids, attention_mask=masks).logits
            logits_1, logits_2 = logits.chunk(2, dim=0)
            cls_loss = 0.5 * (criterion(logits_1, labels) + criterion(logits_2, labels))
            consistency_loss = js_divergence(logits_1, logits_2)
            loss = cls_loss + CFG.lambda_consistency * consistency_loss
            scaled_loss = loss / CFG.grad_accum

        grad_scaler.scale(scaled_loss).backward()

        if step % CFG.grad_accum == 0 or step == len(train_loader):
            grad_scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.max_grad_norm)
            grad_scaler.step(optimizer)
            grad_scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

        running_loss += float(loss.detach().cpu())
        running_cls += float(cls_loss.detach().cpu())
        running_js += float(consistency_loss.detach().cpu())
        progress.set_postfix(loss=f"{running_loss/step:.4f}", js=f"{running_js/step:.4f}")

    val_logits, val_labels, _ = collect_logits(val_loader)
    val_metrics = compute_metrics_from_logits(val_logits, val_labels)
    epoch_row = {
        "epoch": epoch,
        "train_loss": running_loss / len(train_loader),
        "classification_loss": running_cls / len(train_loader),
        "consistency_loss": running_js / len(train_loader),
        **{f"val_{k}": v for k, v in val_metrics.items()},
    }
    history.append(epoch_row)
    print(pd.Series(epoch_row).round(5))

    if val_metrics["macro_f1"] > best_macro_f1 + 1e-6:
        best_macro_f1 = val_metrics["macro_f1"]
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        patience_counter = 0
        print(f"Nouveau meilleur checkpoint : Macro-F1={best_macro_f1:.4f}")
    else:
        patience_counter += 1
        if patience_counter >= CFG.early_stopping_patience:
            print("Early stopping.")
            break

if best_state is None:
    raise RuntimeError("Aucun checkpoint valide n'a été enregistré.")
model.load_state_dict(best_state, strict=True)
model.to(DEVICE).eval()

train_seconds = time.perf_counter() - train_start
peak_vram_gib = (
    torch.cuda.max_memory_allocated() / 2**30 if DEVICE.type == "cuda" else 0.0
)

history_df = pd.DataFrame(history)
history_df.to_csv(Path(CFG.output_dir) / "training_history.csv", index=False)
display(history_df)
print(f"Temps entraînement : {train_seconds/60:.2f} min")
print(f"Pic VRAM           : {peak_vram_gib:.2f} GiB")

## 10 — Baseline numérique optionnelle

La baseline utilise exactement les variables sélectionnées sur le train. Elle n’intervient pas dans la sélection de la méthode SemCons.

In [ ]:
baseline_results = None
if CFG.run_numeric_baseline:
    baseline = HistGradientBoostingClassifier(
        learning_rate=0.08,
        max_iter=250,
        max_leaf_nodes=31,
        l2_regularization=1.0,
        random_state=CFG.seed,
    )
    t0 = time.perf_counter()
    baseline.fit(arrays["train"]["scaled"], arrays["train"]["y"])
    baseline_seconds = time.perf_counter() - t0
    baseline_pred = baseline.predict(arrays["test"]["scaled"])
    baseline_results = {
        "model": "HistGradientBoosting",
        "macro_f1": f1_score(arrays["test"]["y"], baseline_pred, average="macro", zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(arrays["test"]["y"], baseline_pred),
        "mcc": matthews_corrcoef(arrays["test"]["y"], baseline_pred),
        "training_seconds": baseline_seconds,
    }
    print(pd.Series(baseline_results))

## 11 — Temperature scaling

La température est optimisée sur la validation. La partition de calibration reste réservée à la prédiction conforme.

In [ ]:
val_logits, val_labels, _ = collect_logits(val_loader)
cal_logits, cal_labels, cal_indices = collect_logits(cal_loader)


class TemperatureScaler(nn.Module):
    def __init__(self):
        super().__init__()
        self.log_temperature = nn.Parameter(torch.zeros(1))

    @property
    def temperature(self) -> torch.Tensor:
        return self.log_temperature.exp().clamp(0.05, 20.0)

    def forward(self, logits: torch.Tensor) -> torch.Tensor:
        return logits / self.temperature


temp_model = TemperatureScaler().to(DEVICE)
val_logits_t = torch.tensor(val_logits, dtype=torch.float32, device=DEVICE)
val_labels_t = torch.tensor(val_labels, dtype=torch.long, device=DEVICE)
optimizer_t = torch.optim.LBFGS(temp_model.parameters(), lr=0.05, max_iter=100, line_search_fn="strong_wolfe")


def closure():
    optimizer_t.zero_grad()
    loss = F.cross_entropy(temp_model(val_logits_t), val_labels_t)
    loss.backward()
    return loss


optimizer_t.step(closure)
TEMPERATURE = float(temp_model.temperature.detach().cpu())
print(f"Température optimale : {TEMPERATURE:.4f}")

val_before = compute_metrics_from_logits(val_logits, val_labels)
val_after = compute_metrics_from_logits(val_logits / TEMPERATURE, val_labels)
comparison = pd.DataFrame([val_before, val_after], index=["avant", "après"])
display(comparison[["nll", "brier", "ece", "macro_f1", "mcc"]])

## 12 — Prédiction conforme APS et abstention

Le score APS est la masse cumulée jusqu’à la vraie classe dans l’ordre décroissant des probabilités. Le quantile utilise la correction finie `ceil((n+1)(1−α))/n`.

In [ ]:
def softmax_np(logits: np.ndarray) -> np.ndarray:
    shifted = logits - logits.max(axis=1, keepdims=True)
    exp = np.exp(shifted)
    return exp / exp.sum(axis=1, keepdims=True)


def aps_scores(probs: np.ndarray, labels: np.ndarray) -> np.ndarray:
    order = np.argsort(-probs, axis=1)
    sorted_probs = np.take_along_axis(probs, order, axis=1)
    cumulative = np.cumsum(sorted_probs, axis=1)
    positions = np.argmax(order == labels[:, None], axis=1)
    return cumulative[np.arange(len(labels)), positions]


def conformal_quantile(scores: np.ndarray, alpha: float) -> float:
    n = len(scores)
    level = min(1.0, math.ceil((n + 1) * (1 - alpha)) / n)
    return float(np.quantile(scores, level, method="higher"))


def aps_prediction_sets(probs: np.ndarray, qhat: float) -> List[np.ndarray]:
    order = np.argsort(-probs, axis=1)
    sorted_probs = np.take_along_axis(probs, order, axis=1)
    cumulative = np.cumsum(sorted_probs, axis=1)
    sets: List[np.ndarray] = []
    for i in range(len(probs)):
        crossing = int(np.searchsorted(cumulative[i], qhat, side="left"))
        k = min(crossing + 1, probs.shape[1])
        sets.append(order[i, :k])
    return sets


cal_probs = softmax_np(cal_logits / TEMPERATURE)
cal_scores = aps_scores(cal_probs, cal_labels)
QHAT = conformal_quantile(cal_scores, CFG.conformal_alpha)
print(f"q̂ APS (alpha={CFG.conformal_alpha:.2f}) : {QHAT:.6f}")

## 13 — Évaluation finale, couverture et efficacité

In [ ]:
if DEVICE.type == "cuda":
    torch.cuda.synchronize()
t0 = time.perf_counter()
test_logits, test_labels, test_indices = collect_logits(test_loader)
if DEVICE.type == "cuda":
    torch.cuda.synchronize()
inference_seconds = time.perf_counter() - t0

calibrated_test_logits = test_logits / TEMPERATURE
test_probs = softmax_np(calibrated_test_logits)
test_pred = test_probs.argmax(axis=1)
test_metrics = compute_metrics_from_logits(calibrated_test_logits, test_labels)

prediction_sets = aps_prediction_sets(test_probs, QHAT)
coverage = np.mean([label in pred_set for label, pred_set in zip(test_labels, prediction_sets)])
set_sizes = np.array([len(pred_set) for pred_set in prediction_sets])
abstention_rate = float(np.mean(set_sizes > 2))
ambiguity_rate = float(np.mean(set_sizes == 2))
auto_rate = float(np.mean(set_sizes == 1))

latency_ms = 1000 * inference_seconds / len(test_labels)
throughput = len(test_labels) / inference_seconds

final_metrics = {
    **test_metrics,
    "conformal_coverage": float(coverage),
    "average_set_size": float(set_sizes.mean()),
    "singleton_rate": auto_rate,
    "ambiguity_rate": ambiguity_rate,
    "abstention_rate": abstention_rate,
    "train_seconds": train_seconds,
    "peak_vram_gib": peak_vram_gib,
    "inference_seconds": inference_seconds,
    "latency_ms_per_sample": latency_ms,
    "throughput_samples_s": throughput,
    "temperature": TEMPERATURE,
    "conformal_qhat": QHAT,
}

print(pd.Series(final_metrics).round(6))
print("\nRapport de classification :")
print(classification_report(test_labels, test_pred, target_names=class_names, digits=4, zero_division=0))

cm = confusion_matrix(test_labels, test_pred, labels=np.arange(N_CLASSES))
fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(cm, interpolation="nearest")
ax.figure.colorbar(im, ax=ax)
ax.set(
    xticks=np.arange(N_CLASSES), yticks=np.arange(N_CLASSES),
    xticklabels=class_names, yticklabels=class_names,
    xlabel="Prédiction", ylabel="Vérité terrain", title="Matrice de confusion — SemCons-IDS",
)
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
for i in range(N_CLASSES):
    for j in range(N_CLASSES):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=8)
plt.tight_layout()
plt.savefig(Path(CFG.output_dir) / "confusion_matrix.png", dpi=180, bbox_inches="tight")
plt.show()

## 14 — Robustesse aux variantes de sérialisation

In [ ]:
robust_n = min(CFG.robustness_samples, len(arrays["test"]["y"]))
robust_rng = np.random.default_rng(CFG.seed)
robust_indices = np.sort(robust_rng.choice(len(arrays["test"]["y"]), robust_n, replace=False))
robust_dataset = Subset(IndexDataset(arrays["test"]["y"]), robust_indices.tolist())
robust_collator = SemanticBatchCollator(
    tokenizer, arrays["test"]["raw"], arrays["test"]["scaled"], serializer,
    CFG.max_length, dual_view=True,
)
robust_loader = DataLoader(
    robust_dataset, batch_size=CFG.batch_eval, shuffle=False, num_workers=0,
    pin_memory=(DEVICE.type == "cuda"), collate_fn=robust_collator,
)

model.eval()
canon_logits, pert_logits, robust_labels = [], [], []
with torch.no_grad():
    for batch in tqdm(robust_loader, desc="Robustesse"):
        ids1 = batch["input_ids_1"].to(DEVICE)
        mask1 = batch["attention_mask_1"].to(DEVICE)
        ids2 = batch["input_ids_2"].to(DEVICE)
        mask2 = batch["attention_mask_2"].to(DEVICE)
        out1 = model(input_ids=ids1, attention_mask=mask1).logits
        out2 = model(input_ids=ids2, attention_mask=mask2).logits
        canon_logits.append(out1.float().cpu().numpy())
        pert_logits.append(out2.float().cpu().numpy())
        robust_labels.append(batch["labels"].numpy())

canon_logits = np.concatenate(canon_logits) / TEMPERATURE
pert_logits = np.concatenate(pert_logits) / TEMPERATURE
robust_labels = np.concatenate(robust_labels)
canon_pred = canon_logits.argmax(axis=1)
pert_pred = pert_logits.argmax(axis=1)

robustness_results = {
    "n_samples": robust_n,
    "canonical_macro_f1": f1_score(robust_labels, canon_pred, average="macro", zero_division=0),
    "perturbed_macro_f1": f1_score(robust_labels, pert_pred, average="macro", zero_division=0),
    "macro_f1_drop": f1_score(robust_labels, canon_pred, average="macro", zero_division=0)
                     - f1_score(robust_labels, pert_pred, average="macro", zero_division=0),
    "prediction_consistency": float(np.mean(canon_pred == pert_pred)),
    "mean_js_divergence": float(js_divergence(torch.tensor(canon_logits), torch.tensor(pert_logits))),
}
print(pd.Series(robustness_results).round(6))

## 15 — Explicabilité par occlusion des caractéristiques

Pour chaque exemple sélectionné, chaque caractéristique est retirée de la description. L’importance correspond à la baisse de probabilité de la classe prédite.

In [ ]:
@torch.no_grad()
def predict_texts(texts: Sequence[str], batch_size: int = 64) -> np.ndarray:
    outputs = []
    for start in range(0, len(texts), batch_size):
        batch_texts = list(texts[start:start + batch_size])
        enc = tokenizer(
            batch_texts, padding=True, truncation=True, max_length=CFG.max_length,
            pad_to_multiple_of=8, return_tensors="pt",
        )
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        logits = model(**enc).logits / TEMPERATURE
        outputs.append(torch.softmax(logits, dim=-1).cpu().numpy())
    return np.concatenate(outputs)


explain_n = min(CFG.explanation_samples, len(test_labels))
explain_rng = np.random.default_rng(CFG.seed + 7)
explain_indices = np.sort(explain_rng.choice(len(test_labels), explain_n, replace=False))
explanation_rows: List[Dict[str, object]] = []

for local_idx in tqdm(explain_indices, desc="Explications"):
    raw = arrays["test"]["raw"][local_idx]
    scaled = arrays["test"]["scaled"][local_idx]
    base_text = serializer.build(raw, scaled, sample_id=int(local_idx), perturbed=False)
    masked_texts = [
        serializer.build(raw, scaled, sample_id=int(local_idx), perturbed=False, omit_features={j})
        for j in range(len(SELECTED_FEATURES))
    ]
    probs = predict_texts([base_text] + masked_texts)
    predicted = int(probs[0].argmax())
    base_conf = float(probs[0, predicted])
    drops = base_conf - probs[1:, predicted]
    top_order = np.argsort(-drops)[:CFG.explanation_top_k]

    explanation_rows.append({
        "test_index": int(local_idx),
        "true_label": class_names[int(test_labels[local_idx])],
        "predicted_label": class_names[predicted],
        "confidence": base_conf,
        "top_features": " | ".join(SELECTED_FEATURES[j] for j in top_order),
        "importance_drops": " | ".join(f"{float(drops[j]):.6f}" for j in top_order),
        "fidelity_drop_top_k": float(np.clip(drops[top_order], 0, None).sum()),
    })

explanations_df = pd.DataFrame(explanation_rows)
display(explanations_df.head(10))
explanations_df.to_csv(Path(CFG.output_dir) / "feature_occlusion_explanations.csv", index=False)

## 16 — Sauvegarde complète et tableau final

In [ ]:
output_dir = Path(CFG.output_dir)
adapter_dir = output_dir / "best_adapter"
adapter_dir.mkdir(parents=True, exist_ok=True)
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)

# État complet du modèle PEFT, utile pour une reprise exacte dans la même architecture.
torch.save(best_state, output_dir / "best_full_state.pt")

with open(output_dir / "experiment_config.json", "w", encoding="utf-8") as f:
    json.dump(asdict(CFG), f, indent=2, ensure_ascii=False)
with open(output_dir / "reliability.json", "w", encoding="utf-8") as f:
    json.dump({"temperature": TEMPERATURE, "conformal_qhat": QHAT, "alpha": CFG.conformal_alpha}, f, indent=2)

prediction_rows = []
for i, (true_y, pred_y, probs, pred_set) in enumerate(zip(test_labels, test_pred, test_probs, prediction_sets)):
    prediction_rows.append({
        "test_index": int(test_indices[i]),
        "true_label": class_names[int(true_y)],
        "predicted_label": class_names[int(pred_y)],
        "confidence": float(probs[pred_y]),
        "prediction_set": " | ".join(class_names[int(c)] for c in pred_set),
        "set_size": int(len(pred_set)),
        "decision": "automatic" if len(pred_set) == 1 else ("ambiguous" if len(pred_set) == 2 else "abstain"),
    })
predictions_df = pd.DataFrame(prediction_rows)
predictions_df.to_csv(output_dir / "test_predictions_conformal.csv", index=False)

summary = {
    "model": MODEL_CHECKPOINT,
    "backbone": CFG.backbone,
    "dataset": "CICIDS2017",
    "n_classes": N_CLASSES,
    "classes": " | ".join(class_names),
    "selected_features": " | ".join(SELECTED_FEATURES),
    "total_parameters": n_total,
    "lora_parameters": n_lora,
    "head_pooler_parameters": n_head,
    "trainable_parameters": n_trainable,
    "trainable_percent": 100 * n_trainable / n_total,
    **final_metrics,
    **{f"robust_{k}": v for k, v in robustness_results.items()},
}
if baseline_results is not None:
    summary.update({f"baseline_{k}": v for k, v in baseline_results.items()})

summary_df = pd.DataFrame([summary])
summary_df.to_csv(output_dir / "semcons_ids_summary.csv", index=False)
display(summary_df.T)

print("\nPipeline terminé.")
print(f"Adaptateur       : {adapter_dir}")
print(f"Résumé           : {output_dir / 'semcons_ids_summary.csv'}")
print(f"Prédictions      : {output_dir / 'test_predictions_conformal.csv'}")
print(f"Explications     : {output_dir / 'feature_occlusion_explanations.csv'}")

## 17 — Rechargement pour l’inférence

Cette cellule peut être exécutée dans une nouvelle session après avoir relancé les cellules d’installation et d’imports.

In [ ]:
from peft import PeftModel

reload_dir = Path(CFG.output_dir)
reloaded_label_encoder: LabelEncoder = joblib.load(reload_dir / "preprocessing" / "label_encoder.joblib")
reloaded_tokenizer = AutoTokenizer.from_pretrained(reload_dir / "best_adapter")
reloaded_base = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(reloaded_label_encoder.classes_),
    id2label={i: name for i, name in enumerate(reloaded_label_encoder.classes_)},
    label2id={name: i for i, name in enumerate(reloaded_label_encoder.classes_)},
    ignore_mismatched_sizes=True,
    token=HF_TOKEN,
)
reloaded_model = PeftModel.from_pretrained(reloaded_base, reload_dir / "best_adapter").to(DEVICE).eval()

with open(reload_dir / "reliability.json", encoding="utf-8") as f:
    reliability = json.load(f)
print("Modèle rechargé.")
print(reliability)